# InternVL3 Simple Key-Value Extraction

**Purpose:** Simplified single-image extraction demo for InternVL3 vision-language model

This stripped-down notebook:
- Loads a single image
- Displays the image
- Runs InternVL3 extraction
- Shows the extracted key-value pairs

Perfect for quick testing and understanding the extraction process.

In [ ]:
# ============================================================================
# MINIMAL IMPORTS
# ============================================================================

import sys
import warnings
from pathlib import Path
from IPython.display import Image as IPImage, display, Markdown
from PIL import Image

# Add parent directory to Python path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

# Import essential modules
from common.config import (
    INTERNVL3_MODEL_PATH,
    EXTRACTION_FIELDS,
    FIELD_COUNT,
    DATA_DIR
)
from models.internvl3_processor import InternVL3Processor

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

print("✅ Simple InternVL3 Key-Value Extractor Ready")
print(f"📋 Will extract {FIELD_COUNT} fields from business documents")

In [ ]:
# ============================================================================
# INITIALIZE INTERNVL3 MODEL
# ============================================================================

print("🚀 Initializing InternVL3 Model...")
print("=" * 50)

# Initialize processor
processor = InternVL3Processor(model_path=INTERNVL3_MODEL_PATH)

print("\n✅ Model loaded successfully!")
print(f"📍 Model path: {INTERNVL3_MODEL_PATH}")
print(f"🎯 Ready to extract {FIELD_COUNT} business fields")

# Show extraction fields
print("\n📋 Fields to extract:")
for i, field in enumerate(EXTRACTION_FIELDS, 1):
    print(f"  {i:2d}. {field}")
    if i == 5:  # Show first 5 fields then ellipsis
        print(f"  ... and {FIELD_COUNT - 5} more fields")
        break

In [ ]:
# ============================================================================
# SELECT IMAGE TO PROCESS
# ============================================================================

# Default to first sample invoice
# CHANGE THIS PATH to test different images
image_path = Path(DATA_DIR) / "synthetic_invoice_001.jpeg"

# Alternative: specify your own image path
# image_path = Path("/path/to/your/image.jpeg")

# Verify image exists
if not image_path.exists():
    print(f"❌ Image not found: {image_path}")
    print(f"\n📁 Available images in {DATA_DIR}:")
    for img in sorted(Path(DATA_DIR).glob("*.jpeg"))[:5]:
        print(f"   - {img.name}")
else:
    print(f"✅ Selected image: {image_path.name}")
    print(f"📁 Full path: {image_path}")

In [ ]:
# ============================================================================
# DISPLAY THE IMAGE
# ============================================================================

if image_path.exists():
    # Load image with PIL to get dimensions
    pil_image = Image.open(image_path)
    width, height = pil_image.size
    
    print(f"📐 Image dimensions: {width} x {height} pixels")
    print(f"📄 Image type: {pil_image.format}")
    print(f"🎨 Image mode: {pil_image.mode}")
    
    # Display the image
    print("\n🖼️ Document Image:")
    display(IPImage(str(image_path), width=600))  # Adjust width as needed
else:
    print("❌ Cannot display - image file not found")

In [ ]:
# ============================================================================
# RUN EXTRACTION
# ============================================================================

print("🔍 Running InternVL3 Extraction...")
print("=" * 50)

# Process the image
result = processor.process_single_image(str(image_path))

# Extract key information
extracted_data = result['extracted_data']
processing_time = result['processing_time']
extracted_count = result['extracted_fields_count']
raw_response = result['raw_response']

print(f"\n✅ Extraction completed in {processing_time:.2f} seconds")
print(f"📊 Extracted {extracted_count}/{FIELD_COUNT} fields with values")
print(f"📝 Response length: {len(raw_response)} characters")

In [ ]:
# ============================================================================
# DISPLAY EXTRACTED KEY-VALUE PAIRS
# ============================================================================

display(Markdown("## 📋 Extracted Key-Value Pairs"))

# Group fields by extraction status
extracted_fields = {}
missing_fields = []

for field in EXTRACTION_FIELDS:
    value = extracted_data.get(field, "N/A")
    if value != "N/A" and value:
        extracted_fields[field] = value
    else:
        missing_fields.append(field)

# Display extracted fields
if extracted_fields:
    display(Markdown("### ✅ Successfully Extracted Fields:"))
    for field, value in extracted_fields.items():
        # Highlight important fields
        if field in ["TOTAL", "ABN", "INVOICE_NUMBER"]:
            print(f"⭐ {field:<25} : {value}")
        else:
            print(f"   {field:<25} : {value}")

# Display missing fields
if missing_fields:
    display(Markdown("\n### ❌ Fields Not Found:"))
    for field in missing_fields:
        print(f"   {field:<25} : N/A")

# Summary statistics
display(Markdown("\n### 📊 Extraction Summary:"))
print(f"Total fields expected:    {FIELD_COUNT}")
print(f"Fields with values:       {len(extracted_fields)}")
print(f"Missing fields:           {len(missing_fields)}")
print(f"Extraction rate:          {(len(extracted_fields)/FIELD_COUNT)*100:.1f}%")
print(f"Processing time:          {processing_time:.2f} seconds")

In [ ]:
# ============================================================================
# OPTIONAL: VIEW RAW RESPONSE
# ============================================================================

# Uncomment the lines below to see the raw model response

# display(Markdown("## 📝 Raw Model Response"))
# print("Raw output from InternVL3:")
# print("=" * 50)
# print(raw_response)
# print("=" * 50)

In [ ]:
# ============================================================================
# QUICK TEST WITH DIFFERENT IMAGE
# ============================================================================

# This cell allows you to quickly test another image without re-running everything

def quick_extract(image_file):
    """Quick extraction function for testing multiple images."""
    print(f"\n🔍 Processing: {Path(image_file).name}")
    print("=" * 40)
    
    # Check if file exists
    if not Path(image_file).exists():
        print(f"❌ File not found: {image_file}")
        return
    
    # Process image
    result = processor.process_single_image(str(image_file))
    
    # Display key results
    extracted_data = result['extracted_data']
    extracted_count = sum(1 for v in extracted_data.values() if v != "N/A")
    
    print(f"✅ Extracted {extracted_count}/{FIELD_COUNT} fields")
    print(f"⏱️  Time: {result['processing_time']:.2f}s\n")
    
    # Show top fields
    print("Key fields:")
    for field in ["TOTAL", "ABN", "INVOICE_NUMBER", "SUPPLIER_NAME", "INVOICE_DATE"]:
        value = extracted_data.get(field, "N/A")
        if value != "N/A":
            print(f"  {field}: {value}")
    
    return extracted_data

# Example: Test with another invoice
# Uncomment and modify the path below to test
# test_result = quick_extract(Path(DATA_DIR) / "synthetic_invoice_002.jpeg")